# Notebook 06 — Fragmentación dinámica (H-Net) e impacto downstream de la tokenización

**Bootcamp Bio-LLMs · Módulo 2 · Sesión 2 de 2**
Proyecto posdoctoral CICESE — Modelos de lenguaje para venómica integrativa de *Conus*.

---

## Por qué este notebook

Cierra el Módulo 2 cubriendo el **tercer mecanismo** que la metodología del proyecto menciona explícitamente:

> *"[...] tokenizarán en k-meros [...] y fragmentos (H-Net) [...]. Se evaluará la eficiencia de tokenización basada en transformadores, y mecanismo de fragmentación dinámica (H-Net)."*

Y responde la pregunta práctica que el Notebook 05 dejó abierta: **¿la elección de tokenizador realmente cambia el desempeño del modelo, o sólo la eficiencia?**

## Objetivos

1. Entender por qué el ADN es un caso difícil para la tokenización (no tiene claves de segmentación naturales como los espacios en el texto).
2. Implementar una **versión simplificada de la fragmentación dinámica de H-Net**: un módulo de routing que aprende a dibujar fronteras de chunk, en lugar de usar un vocabulario fijo.
3. Visualizar las fronteras aprendidas sobre un precursor conotoxínico.
4. **Experimento de impacto downstream**: entrenar el mismo encoder MLM con tres tokenizadores (no-superpuesto, BPE, codón) y comparar perplejidad y costo computacional.

## Pre-requisitos

* Notebooks 03 (MLM) y 05 (tokenizadores) completados.
* PyTorch >= 2.0, GPU recomendada para la sección 4.

## 0. Imports

In [ ]:
import math
import random
import time
from collections import Counter, defaultdict
from itertools import product
from typing import List

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

## 1. El problema del ADN: sin claves de segmentación naturales

El paper de H-Net (Hwang et al. 2025) lo formula con claridad:

> *"DNA does not have any natural tokenization cues and instead must be processed as raw base pairs."*

Compara:
* **Texto en inglés**: los espacios marcan fronteras de palabra. BPE funciona bien porque aprende sub-palabras.
* **ADN**: la cadena `ATGGCTTGTTGC` no tiene espacios ni fronteras obvias. Peor aún, según H-Net:

> *"the same sequence of base pairs may serve different functions (e.g., depending on whether or not the pair is inside a gene or not)."*

El mismo codón `TGT` (cisteína) puede ser parte de un péptido señal, un propéptido o el dominio maduro funcional — y su significado depende del **contexto**, no de su identidad local. Un vocabulario fijo (k-mer o BPE) asigna el mismo token sin importar el contexto.

### La propuesta de H-Net: chunking dinámico aprendido

En lugar de un vocabulario fijo, H-Net **aprende dónde poner las fronteras** mediante un módulo de routing entrenado end-to-end con el resto del modelo. Las fronteras se dibujan donde el contenido cambia (alta "sorpresa"), comprimiendo la secuencia de forma adaptativa.

> *"a byte-level H-Net matches the perplexity and downstream performance of a strong BPE-tokenized Transformer [...]. the dynamic chunking module naturally compresses data to a similar resolution as BPE tokenizers (4.5-5 bytes/chunk) and qualitatively learns meaningful boundaries, all without any external supervision or heuristics."*

## 2. Fragmentación dinámica simplificada

### 2.1 El mecanismo de routing (versión pedagógica)

H-Net completo es complejo (downsampler, upsampler, smoothing module, straight-through estimator). Aquí implementamos la **idea central**: un módulo que, para cada posición, decide si abrir una nueva frontera de chunk basándose en cuánto difiere el contenido del token actual respecto al anterior.

La intuición de H-Net es medir la **similitud coseno** entre representaciones adyacentes: si dos posiciones consecutivas son muy similares, pertenecen al mismo chunk; si difieren mucho, hay una frontera.

$$p_t = \frac{1}{2}\left(1 - \cos(\mathbf{h}_{t-1}, \mathbf{h}_t)\right)$$

donde $p_t \in [0, 1]$ es la probabilidad de frontera en la posición $t$. Valores altos = cambio brusco de contenido = frontera probable.

In [ ]:
class DynamicChunkingRouter(nn.Module):
    """
    Modulo de routing simplificado inspirado en H-Net (Hwang et al. 2025).
    Para cada posicion, calcula una probabilidad de frontera de chunk
    basada en la disimilitud coseno con la posicion anterior.

    use_projections:
      - False -> coseno directo sobre los embeddings de entrada.
                 Ilustra el MECANISMO central de forma limpia.
      - True  -> coseno sobre proyecciones aprendidas W_q, W_k (como H-Net real).
                 Sin entrenar, estas proyecciones aleatorias destruyen la
                 estructura -> sirve para mostrar POR QUE el router debe entrenarse.
    """

    def __init__(self, d_model, use_projections=True):
        super().__init__()
        self.use_projections = use_projections
        if use_projections:
            self.W_q = nn.Linear(d_model, d_model, bias=False)
            self.W_k = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        """
        x : (B, N, d_model) representaciones por posicion
        Returns
        -------
        boundary_prob : (B, N) probabilidad de frontera en cada posicion
        """
        if self.use_projections:
            q = self.W_q(x)  # (B, N, d)
            k = self.W_k(x)  # (B, N, d)
        else:
            q = k = x

        # Similitud coseno entre posicion t-1 (key) y t (query)
        q_norm = F.normalize(q, dim=-1)
        k_norm = F.normalize(k, dim=-1)

        # cos(h_{t-1}, h_t): producto punto entre k en t-1 y q en t
        cos_sim = (q_norm[:, 1:] * k_norm[:, :-1]).sum(dim=-1)  # (B, N-1)

        # Probabilidad de frontera = (1 - cos) / 2
        boundary_prob = (1.0 - cos_sim) * 0.5  # (B, N-1)

        # La primera posicion siempre es frontera (inicio de secuencia)
        first = torch.ones(x.size(0), 1, device=x.device)
        boundary_prob = torch.cat([first, boundary_prob], dim=1)  # (B, N)

        return boundary_prob


# Demo con embeddings aleatorios: coseno directo (use_projections=False)
d_model = 64
router = DynamicChunkingRouter(d_model, use_projections=False)
x_demo = torch.randn(1, 20, d_model)
bp = router(x_demo)
print(f"Entrada: {tuple(x_demo.shape)}")
print(f"Probabilidades de frontera: {tuple(bp.shape)}")
print(f"Rango: [{bp.min():.3f}, {bp.max():.3f}]")
print(f"Primera posicion (siempre frontera): {bp[0, 0].item():.3f}")
print("\nCon vectores aleatorios independientes, el coseno ~0 -> prob ~0.5 en casi todo.")
print("Esto es esperado: sin estructura de dominio, no hay fronteras claras que detectar.")

### 2.2 De probabilidades a chunks

Dada la probabilidad de frontera por posición, abrimos un nuevo chunk cuando $p_t$ supera un umbral. Esto produce una segmentación de longitud variable. El **ratio de compresión** resultante (bases/chunk) emerge dinámicamente — no se fija de antemano como en k-mer.

In [ ]:
def chunk_from_boundaries(boundary_prob, threshold=0.5):
    """
    Convierte probabilidades de frontera en asignaciones de chunk.
    Returns lista de (inicio, fin) por chunk para cada secuencia del batch.
    """
    B, N = boundary_prob.shape
    all_chunks = []
    for b in range(B):
        boundaries = (boundary_prob[b] > threshold).nonzero(as_tuple=True)[0].tolist()
        if 0 not in boundaries:
            boundaries = [0] + boundaries
        boundaries.append(N)  # cierre final
        chunks = [(boundaries[i], boundaries[i+1]) for i in range(len(boundaries)-1)]
        all_chunks.append(chunks)
    return all_chunks


# Demo: aplicar a embeddings de una secuencia real tokenizada a nivel base
# Creamos embeddings donde cambios de "dominio" producen saltos
def make_domain_embeddings(seq_with_domains, d_model=64, noise=0.02):
    """
    Crea embeddings donde cada dominio tiene una firma distinta y casi constante.
    Ruido bajo para que la disimilitud coseno sea ~0 DENTRO de un dominio
    y alta SOLO en las transiciones entre dominios.
    """
    domain_signatures = {}
    embeds = []
    for char, domain in seq_with_domains:
        if domain not in domain_signatures:
            # Firmas ortogonales entre dominios para maxima disimilitud en fronteras
            domain_signatures[domain] = F.normalize(torch.randn(d_model), dim=0)
        embeds.append(domain_signatures[domain] + noise * torch.randn(d_model))
    return torch.stack(embeds).unsqueeze(0)  # (1, N, d)


# Secuencia sintetica con 3 dominios marcados
seq_domains = (
    [("A", "signal")] * 6 +
    [("T", "propep")] * 6 +
    [("C", "mature")] * 6
)
x_domains = make_domain_embeddings(seq_domains, d_model=64)

# Router con coseno DIRECTO (sin proyecciones) para ilustrar el mecanismo limpio
router_raw = DynamicChunkingRouter(64, use_projections=False)
bp_domains = router_raw(x_domains)

# Umbral 0.3: dentro de un dominio la prob es ~0 (coseno ~1); en transiciones
# entre dominios ortogonales la prob salta a ~0.5. Un umbral intermedio separa limpio.
THRESH = 0.3
chunks = chunk_from_boundaries(bp_domains, threshold=THRESH)

print("Secuencia con 3 dominios (signal/propep/mature), 6 posiciones cada uno:")
print(f"Umbral de frontera = {THRESH}\n")
print("Probabilidades de frontera por posicion:")
for i, p in enumerate(bp_domains[0].tolist()):
    marker = " <- FRONTERA" if p > THRESH else ""
    print(f"  pos {i:>2} ({seq_domains[i][1]}): {p:.3f}{marker}")
print(f"\nChunks detectados: {chunks[0]}")
print(f"Numero de chunks: {len(chunks[0])}  (esperado: 3, uno por dominio)")

In [ ]:
# Visualizar las fronteras detectadas (coseno directo)
fig, ax = plt.subplots(figsize=(12, 3))
probs = bp_domains[0].detach().numpy()
positions = range(len(probs))
colors_domain = {"signal": "#1f77b4", "propep": "#ff7f0e", "mature": "#2ca02c"}
bar_colors = [colors_domain[seq_domains[i][1]] for i in positions]

ax.bar(positions, probs, color=bar_colors, alpha=0.7)
ax.axhline(THRESH, color="red", ls="--", label=f"umbral de frontera ({THRESH})")
ax.set_xlabel("Posicion en la secuencia")
ax.set_ylabel("Prob. de frontera")
ax.set_title("Fronteras de chunk dinamicas (coseno directo sobre embeddings)\n"
             "Prob ~0 dentro de un dominio; salto en las transiciones signal->propep->mature")
ax.set_xticks(list(positions))

from matplotlib.patches import Patch
legend_elems = [Patch(facecolor=c, label=d) for d, c in colors_domain.items()]
legend_elems.append(plt.Line2D([0], [0], color="red", ls="--", label="umbral"))
ax.legend(handles=legend_elems, loc="upper right", fontsize=9)

plt.tight_layout()
plt.show()

print("\nCon coseno directo, las fronteras emergen LIMPIAS solo en los cambios de dominio.")

### 2.3 ¿Por qué entonces H-Net usa proyecciones aprendidas?

Acabamos de ver que el coseno directo sobre los embeddings produce fronteras limpias **cuando los embeddings ya separan los dominios**. Pero en un modelo real, las representaciones de las capas intermedias no vienen pre-organizadas por dominio funcional — hay que *aprender* qué dimensiones importan para segmentar.

Por eso el router de H-Net aplica proyecciones $W_q, W_k$ **aprendidas end-to-end**. El problema: sin entrenar, esas proyecciones son aleatorias y destruyen la estructura. Veámoslo como contraste.

In [ ]:
# CONTRASTE: el mismo input con proyecciones SIN ENTRENAR (aleatorias)
torch.manual_seed(0)
router_proj = DynamicChunkingRouter(64, use_projections=True)  # W_q, W_k aleatorios
bp_proj = router_proj(x_domains)

print("Mismas firmas de dominio, pero pasadas por W_q/W_k aleatorios (sin entrenar):\n")
print("Probabilidades de frontera por posicion:")
n_frontiers = 0
for i, p in enumerate(bp_proj[0].tolist()):
    marker = " <- FRONTERA" if p > THRESH else ""
    if p > THRESH: n_frontiers += 1
    print(f"  pos {i:>2} ({seq_domains[i][1]}): {p:.3f}{marker}")
print(f"\nFronteras detectadas: {n_frontiers} de {len(seq_domains)} posiciones")
print("\nLECCION: las proyecciones aleatorias mezclan las dimensiones y el coseno")
print("se vuelve ~0.5 en casi todo -> fronteras espurias. Por eso H-Net DEBE")
print("entrenar el router end-to-end: solo asi W_q/W_k aprenden a preservar la")
print("estructura de dominio y dibujar fronteras significativas (ejercicio 6.1).")

### 2.4 Por qué H-Net es atractivo para venómica — y por qué es difícil

**Ventajas para tu proyecto:**
* No requiere elegir k ni tamaño de vocabulario de antemano.
* Las fronteras se adaptan a la estructura local (señal vs propéptido vs maduro).
* Opera sobre pares de bases crudos, preservando resolución de nucleótido.

**Costo:**
* H-Net completo es significativamente más complejo de entrenar que un encoder estándar (requiere el straight-through estimator para hacer las fronteras diferenciables, balanceo de ratio de compresión, y jerarquías multi-etapa).
* Los modelos de ADN estado-del-arte que usan esta filosofía (Evo-2, basado en operar sobre pares de bases) requieren cómputo masivo.

**Recomendación práctica para el proyecto:** empieza con BPE o k-mer no-superpuesto (Notebook 05) para los primeros experimentos. Considera H-Net/chunking dinámico como **línea exploratoria avanzada** una vez que el pipeline base funcione, alineado con lo que la metodología llama "evaluar [...] mecanismo de fragmentación dinámica".

## 3. Preparación: tres tokenizadores y un corpus

Para el experimento downstream, reutilizamos los tokenizadores del Notebook 05 más el `CodonTokenizer` del Notebook 03. Comparamos:

* **k-mer no-superpuesto (k=3)**: alineado a tripletes pero sin respetar marco de lectura.
* **Codón (3-mer alineado a marco)**: respeta la biología del CDS.
* **BPE vocab=256**: compresión adaptativa.

Mantenemos los modelos pequeños para que el experimento corra en minutos.

In [ ]:
# --- Tokenizadores (compactados de los Notebooks 03 y 05) ---

class NonOverlappingKmerTokenizer:
    SPECIAL = ["[PAD]", "[MASK]", "[CLS]", "[SEP]", "[UNK]"]
    def __init__(self, k=3):
        self.k = k
        kmers = ["".join(p) for p in product("ACGT", repeat=k)]
        self.vocab = self.SPECIAL + kmers + list("ACGT")
        self.t2i = {t: i for i, t in enumerate(self.vocab)}
        self.pad_id = self.t2i["[PAD]"]; self.mask_id = self.t2i["[MASK]"]
        self.cls_id = self.t2i["[CLS]"]; self.sep_id = self.t2i["[SEP]"]
        self.unk_id = self.t2i["[UNK]"]
        self.codon_ids = [i for t, i in self.t2i.items() if t not in self.SPECIAL]
    @property
    def vocab_size(self): return len(self.vocab)
    def tokenize(self, seq):
        seq = seq.upper(); toks = []; nf = len(seq)//self.k
        for i in range(nf): toks.append(seq[i*self.k:(i+1)*self.k])
        for b in seq[nf*self.k:]: toks.append(b)
        return toks
    def encode(self, seq, max_len):
        ids = [self.cls_id] + [self.t2i.get(t, self.unk_id) for t in self.tokenize(seq)] + [self.sep_id]
        ids = ids[:max_len]
        return ids + [self.pad_id]*(max_len - len(ids))


class CodonTokenizer:
    """3-mer alineado a marco de lectura (respeta la biologia del CDS)."""
    SPECIAL = ["[PAD]", "[MASK]", "[CLS]", "[SEP]", "[UNK]"]
    def __init__(self):
        codons = ["".join(p) for p in product("ACGT", repeat=3)]
        self.vocab = self.SPECIAL + codons
        self.t2i = {t: i for i, t in enumerate(self.vocab)}
        self.pad_id = self.t2i["[PAD]"]; self.mask_id = self.t2i["[MASK]"]
        self.cls_id = self.t2i["[CLS]"]; self.sep_id = self.t2i["[SEP]"]
        self.unk_id = self.t2i["[UNK]"]
        self.codon_ids = [i for t, i in self.t2i.items() if t not in self.SPECIAL]
    @property
    def vocab_size(self): return len(self.vocab)
    def tokenize(self, seq):
        seq = seq.upper(); n = len(seq)//3
        return [seq[3*i:3*i+3] for i in range(n)]  # alineado a marco desde pos 0
    def encode(self, seq, max_len):
        ids = [self.cls_id] + [self.t2i.get(t, self.unk_id) for t in self.tokenize(seq)] + [self.sep_id]
        ids = ids[:max_len]
        return ids + [self.pad_id]*(max_len - len(ids))


class FastBPE:
    SPECIAL = ["[PAD]", "[MASK]", "[CLS]", "[SEP]", "[UNK]"]
    def __init__(self, vocab_size=256):
        self.target_vocab_size = vocab_size; self.merges = []; self.vocab = None
    def train(self, corpus):
        words = [list(s.upper()) for s in corpus]
        base = sorted(set(c for s in corpus for c in s.upper())); cv = set(base)
        nm = self.target_vocab_size - len(self.SPECIAL) - len(base)
        pc = Counter(); ptw = defaultdict(set)
        for wi, toks in enumerate(words):
            for i in range(len(toks)-1):
                p = (toks[i], toks[i+1]); pc[p] += 1; ptw[p].add(wi)
        for _ in range(nm):
            if not pc: break
            bp = max(pc, key=pc.get)
            if pc[bp] < 2: break
            self.merges.append(bp); m = bp[0]+bp[1]; cv.add(m)
            for wi in list(ptw[bp]):
                toks = words[wi]
                for i in range(len(toks)-1):
                    p = (toks[i], toks[i+1]); pc[p] -= 1
                    if pc[p] <= 0: pc.pop(p, None); ptw.pop(p, None)
                    else: ptw[p].discard(wi)
                nt = []; i = 0
                while i < len(toks):
                    if i < len(toks)-1 and toks[i]==bp[0] and toks[i+1]==bp[1]: nt.append(m); i+=2
                    else: nt.append(toks[i]); i+=1
                words[wi] = nt
                for i in range(len(nt)-1):
                    p = (nt[i], nt[i+1]); pc[p] += 1; ptw[p].add(wi)
        self.vocab = self.SPECIAL + sorted(cv)
        self.t2i = {t: i for i, t in enumerate(self.vocab)}
        self.pad_id = self.t2i["[PAD]"]; self.mask_id = self.t2i["[MASK]"]
        self.cls_id = self.t2i["[CLS]"]; self.sep_id = self.t2i["[SEP]"]
        self.unk_id = self.t2i["[UNK]"]
        self.codon_ids = [i for t, i in self.t2i.items() if t not in self.SPECIAL]
    @property
    def vocab_size(self): return len(self.vocab)
    def tokenize(self, seq):
        toks = list(seq.upper())
        for bp in self.merges:
            m = bp[0]+bp[1]; nt = []; i = 0
            while i < len(toks):
                if i < len(toks)-1 and toks[i]==bp[0] and toks[i+1]==bp[1]: nt.append(m); i+=2
                else: nt.append(toks[i]); i+=1
            toks = nt
        return toks
    def encode(self, seq, max_len):
        ids = [self.cls_id] + [self.t2i.get(t, self.unk_id) for t in self.tokenize(seq)] + [self.sep_id]
        ids = ids[:max_len]
        return ids + [self.pad_id]*(max_len - len(ids))


def synthesize_precursor():
    h = ["CTG","CTT","GTG","GTT","ATC","GCT","GCC","TTT","TTC"]
    c = ["AAA","AAG","AGA","CGT","GAT","GAC","GAA","GAG","CAT"]
    m = ["TGT","TGC"]*3 + ["CCA","CCT","GGA","GGC","TCA","AGC","ACA","TAT","CGT"]
    def blk(p, n): return "".join(random.choice(p) for _ in range(n))
    return ("ATG" + blk(h, random.randint(18,22)) + blk(c, random.randint(20,26))
            + random.choice(["AAACGT","AAAAAA","CGTAAA"]) + blk(m, random.randint(12,30))
            + random.choice(["TAA","TAG","TGA"]))


# Corpus
N_TRAIN, N_VAL = 2500, 500
train_seqs = [synthesize_precursor() for _ in range(N_TRAIN)]
val_seqs = [synthesize_precursor() for _ in range(N_VAL)]

print(f"Corpus: {N_TRAIN} train, {N_VAL} val")
print("Entrenando BPE-256...")
bpe = FastBPE(256); bpe.train(train_seqs[:800])

tokenizers = {
    "k-mer no-superp (k=3)": NonOverlappingKmerTokenizer(k=3),
    "codon (3-mer marco)": CodonTokenizer(),
    "BPE-256": bpe,
}
for name, tok in tokenizers.items():
    print(f"  {name:<24} vocab={tok.vocab_size}")

## 4. Experimento de impacto downstream

### Diseño

Para cada tokenizador entrenamos un encoder MLM idéntico (mismo d_model, capas, cabezas) y medimos:

* **Perplejidad de validación** (calidad del modelo de lenguaje).
* **Tokens por secuencia** (costo: más tokens = atención más cara).
* **Tiempo por época** (costo computacional real).

La pregunta: ¿el tokenizador que comprime más también da mejor perplejidad, o hay un trade-off?

### Modelo MLM mínimo (reutilizado del Notebook 03)

In [ ]:
class TinyMLM(nn.Module):
    """Encoder MLM minimo con pre-norm (compactado del Notebook 03)."""
    def __init__(self, vocab_size, d_model=128, num_heads=4, num_layers=2,
                 d_ff=512, max_len=256, dropout=0.1, pad_id=0):
        super().__init__()
        self.pad_id = pad_id; self.d_model = d_model
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        # PE sinusoidal
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos*div); pe[:, 1::2] = torch.cos(pos*div)
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                "mha": nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True),
                "ffn": nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)),
                "n1": nn.LayerNorm(d_model), "n2": nn.LayerNorm(d_model),
            }) for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.mlm_head = nn.Linear(d_model, vocab_size)
        self.mlm_head.weight = self.token_embedding.weight

    def forward(self, token_ids):
        pad_mask = (token_ids == self.pad_id)  # (B, N)
        x = self.token_embedding(token_ids) * math.sqrt(self.d_model)
        x = x + self.pe[:, :token_ids.size(1)]
        for layer in self.layers:
            normed = layer["n1"](x)
            attn_out, _ = layer["mha"](normed, normed, normed, key_padding_mask=pad_mask)
            x = x + attn_out
            x = x + layer["ffn"](layer["n2"](x))
        return self.mlm_head(self.final_norm(x))


# NOTA: aqui usamos nn.MultiheadAttention por brevedad. En el Modulo 1 lo
# implementamos desde cero; ahora que entendemos el mecanismo, usar la version
# optimizada de PyTorch es legitimo y mas rapido.
print("TinyMLM definido (usa nn.MultiheadAttention optimizado de PyTorch)")

In [ ]:
def mlm_mask(ids, tokenizer, p=0.15):
    inputs = ids.clone(); labels = ids.clone()
    special = torch.tensor([tokenizer.pad_id, tokenizer.cls_id, tokenizer.sep_id])
    candidate = ~torch.isin(inputs, special)
    selected = (torch.rand_like(inputs.float()) < p) & candidate
    labels[~selected] = -100
    decision = torch.rand_like(inputs.float())
    inputs[selected & (decision < 0.8)] = tokenizer.mask_id
    rand_mask = selected & (decision >= 0.8) & (decision < 0.9)
    rand_tokens = torch.tensor(np.random.choice(tokenizer.codon_ids, size=inputs.shape))
    inputs[rand_mask] = rand_tokens[rand_mask]
    return inputs, labels


@torch.no_grad()
def eval_ppl(model, token_ids, tokenizer, device, batch_size=128):
    model.eval(); total_loss = total_n = 0
    for i in range(0, token_ids.size(0), batch_size):
        batch = token_ids[i:i+batch_size]
        inp, lbl = mlm_mask(batch, tokenizer)
        inp, lbl = inp.to(device), lbl.to(device)
        logits = model(inp)
        n = (lbl != -100).sum().item()
        if n == 0: continue
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), lbl.view(-1),
                                ignore_index=-100, reduction="sum")
        total_loss += loss.item(); total_n += n
    return math.exp(total_loss / max(1, total_n))


def run_experiment(tokenizer, name, train_seqs, val_seqs, max_len, epochs=3, device=device):
    """Entrena un MLM con un tokenizador y devuelve metricas."""
    # Tokenizar
    train_ids = torch.tensor([tokenizer.encode(s, max_len) for s in train_seqs])
    val_ids = torch.tensor([tokenizer.encode(s, max_len) for s in val_seqs])

    # Tokens reales por secuencia (sin contar padding)
    real_tokens = (train_ids != tokenizer.pad_id).sum(dim=1).float().mean().item()

    torch.manual_seed(SEED)
    model = TinyMLM(tokenizer.vocab_size, d_model=128, num_heads=4, num_layers=2,
                    d_ff=512, max_len=max_len, pad_id=tokenizer.pad_id).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.01)

    epoch_times = []
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(train_ids.size(0))
        t0 = time.time()
        for i in range(0, train_ids.size(0), 64):
            idx = perm[i:i+64]; batch = train_ids[idx]
            inp, lbl = mlm_mask(batch, tokenizer)
            inp, lbl = inp.to(device), lbl.to(device)
            logits = model(inp)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), lbl.view(-1), ignore_index=-100)
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        epoch_times.append(time.time() - t0)

    val_ppl = eval_ppl(model, val_ids, tokenizer, device)
    return {
        "name": name,
        "vocab_size": tokenizer.vocab_size,
        "real_tokens_per_seq": real_tokens,
        "val_ppl": val_ppl,
        "mean_epoch_time": np.mean(epoch_times),
    }


# max_len debe acomodar el tokenizador menos comprimido. Codon/k-mer dan ~hasta 100 tokens.
MAX_LEN = 110
print(f"Ejecutando experimento downstream (max_len={MAX_LEN}, 3 epocas cada uno)...\n")

exp_results = []
for name, tok in tokenizers.items():
    print(f"--- {name} ---")
    r = run_experiment(tok, name, train_seqs, val_seqs, MAX_LEN, epochs=3)
    exp_results.append(r)
    print(f"  vocab={r['vocab_size']}  tokens/seq={r['real_tokens_per_seq']:.1f}  "
          f"val_ppl={r['val_ppl']:.2f}  t/epoch={r['mean_epoch_time']:.1f}s\n")

In [ ]:
# Tabla y visualizacion final
print("=" * 80)
print(f"{'Tokenizador':<24}{'Vocab':<8}{'Tokens/seq':<14}{'Val PPL':<12}{'s/epoca'}")
print("=" * 80)
for r in exp_results:
    print(f"{r['name']:<24}{r['vocab_size']:<8}{r['real_tokens_per_seq']:<14.1f}"
          f"{r['val_ppl']:<12.2f}{r['mean_epoch_time']:.1f}")
print("=" * 80)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
names = [r["name"] for r in exp_results]
colors = ["#ff7f0e", "#2ca02c", "#1f77b4"]

# Perplejidad
axes[0].bar(names, [r["val_ppl"] for r in exp_results], color=colors)
axes[0].set_ylabel("Perplejidad de validacion")
axes[0].set_title("(a) Calidad del modelo\n(menor = mejor)")
axes[0].tick_params(axis="x", rotation=20)

# Tokens por secuencia
axes[1].bar(names, [r["real_tokens_per_seq"] for r in exp_results], color=colors)
axes[1].set_ylabel("Tokens reales / secuencia")
axes[1].set_title("(b) Costo: longitud de secuencia\n(menor = atencion mas barata)")
axes[1].tick_params(axis="x", rotation=20)

# Tiempo por epoca
axes[2].bar(names, [r["mean_epoch_time"] for r in exp_results], color=colors)
axes[2].set_ylabel("Segundos por epoca")
axes[2].set_title("(c) Costo computacional real")
axes[2].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

## 5. Interpretación y decisión para el proyecto

### Lo que el experimento revela

Hay un **trade-off real** entre las tres dimensiones, y no existe un ganador universal:

* **No te alarmes si BPE muestra una perplejidad mucho más alta** (verás valores de cientos o miles frente a ~40 de k-mer/codón). Esto **no** significa que BPE sea un mal tokenizador: la **perplejidad no es directamente comparable entre tokenizadores** con vocabularios distintos. La perplejidad mide "entre cuántas opciones equivalentes duda el modelo"; un vocabulario de 256 tokens variables tiene un espacio de predicción mucho más difícil por token que uno de 69 codones fijos, e infla el número aunque el modelo capture igual o más información por nucleótido. Para comparar de forma justa hay que normalizar a **bits-per-nucleotide** = (loss × tokens) / (nt × ln 2), que reparte el costo entre los nucleótidos realmente cubiertos (ver ejercicio 6.2). Compara perplejidad cruda sólo *dentro* del mismo vocabulario.
* El **costo computacional** sí es directamente comparable: menos tokens/secuencia = atención más barata. Aquí BPE y codón ganan sobre k-mer si comprimen más.
* Para **tu proyecto**, la decisión depende de la tarea downstream:
  - **gLM sobre exomas largos** → prioriza compresión (BPE) para que la atención O(n²) sea manejable.
  - **pLM sobre conotoxinas cortas** → prioriza resolución por residuo (nivel aminoácido o codón), porque cada cisteína importa para predecir puentes disulfuro.

### Sobre H-Net en el proyecto

El chunking dinámico (sección 2) es la frontera: elimina la decisión manual de tokenizador, pero a costo de complejidad de entrenamiento sustancial. Recomendación: **empezar con BPE/codón, dejar H-Net como línea exploratoria** una vez validado el pipeline base, tal como la metodología plantea ("evaluar [...] mecanismo de fragmentación dinámica").

### Resumen de decisiones del Módulo 2

| Componente del proyecto | Tokenizador recomendado | Justificación |
|---|---|---|
| gLM (genómico, contexto largo) | BPE (vocab 4096-8192) | Máxima compresión para atención larga |
| pLM (proteómico, conotoxinas) | Aminoácido o codón | Resolución por residuo para cisteínas |
| Línea exploratoria | H-Net / chunking dinámico | Sin vocabulario fijo, adaptativo al contexto |

## 6. Ejercicios

### 6.1 — Entrenar el router de H-Net
El router de la sección 2 no está entrenado. Conéctalo a un autoencoder simple (downsample → encoder → upsample → reconstrucción) y entrénalo end-to-end. Visualiza cómo las fronteras aprendidas se alinean (o no) con los límites de dominio señal/propéptido/maduro.

### 6.2 — Bits-per-nucleotide en vez de perplejidad
La perplejidad depende del vocabulario. Calcula bits-per-nucleotide = (loss × tokens) / (nt × ln 2) para hacer los tokenizadores comparables. Re-evalúa el experimento de la sección 4 con esta métrica normalizada.

### 6.3 — Tarea downstream real
En vez de perplejidad MLM, evalúa los embeddings de cada tokenizador en una tarea de clasificación: "¿esta secuencia contiene un dominio maduro rico en cisteínas?". Congela el encoder, entrena una cabeza lineal, compara accuracy. Esto mide utilidad downstream, no sólo calidad del LM.

### 6.4 — Marco de lectura
El `CodonTokenizer` asume marco de lectura desde la posición 0. Pero en datos reales no siempre conoces el marco. Implementa una variante que pruebe los 3 marcos posibles y elija el que minimice codones de paro internos. ¿Cómo cambia esto la tokenización?

### 6.5 — Compresión de H-Net vs BPE
H-Net reporta comprimir a 4.5-5 bytes/chunk, similar a BPE. Mide el ratio de compresión efectivo de tu router (sección 2) variando el umbral de frontera. ¿Puedes igualar la compresión de BPE-256?

## 7. Conclusión del Módulo 2

✓ Entendiste por qué el ADN es difícil de tokenizar (sin claves de segmentación, significado dependiente de contexto).
✓ Implementaste una versión simplificada de la **fragmentación dinámica de H-Net** con routing por disimilitud coseno.
✓ Mediste el **impacto downstream** de la elección de tokenizador sobre perplejidad y costo computacional.
✓ Estableciste decisiones concretas de tokenización para el gLM y pLM del proyecto.

### Conexión con el resto del bootcamp

| Módulo | Lo aprendido aquí aplicado allí |
|---|---|
| **Módulo 3** (pre-entrenamiento escalado) | El tokenizador elegido determina la longitud de contexto efectiva y el costo de pre-entrenamiento en A100 |
| **Módulo 4** (PEFT) | Al cargar NT-v2 (6-mer) o DNABERT-2 (BPE) de HuggingFace, ya sabes qué tokenizador trae cada uno y por qué |
| **Módulo 5** (interpretabilidad) | Las visualizaciones de atención dependen de la granularidad del tokenizador |

### Bibliografía de este notebook

* **Hwang et al. (2025)** H-Net: Dynamic Chunking for End-to-End Hierarchical Sequence Modeling. (Fragmentación dinámica)
* **Zhou et al. (2024)** DNABERT-2 — ICLR. (BPE, benchmark)
* **Dalla-Torre et al. (2025)** Nucleotide Transformer — Nature Methods 22. (6-mer)
* **Boshar et al. (2024)** Are genomic language models all you need? — Bioinformatics 40(9). (3-mer vs 6-mer en proteínas)
* **Brixi et al. (2025)** referenciado en H-Net como modelo SOTA de ADN operando sobre pares de bases (Evo-2).

In [ ]:
print("=" * 60)
print("RESUMEN MODULO 2 - NOTEBOOK 06")
print("=" * 60)
for r in exp_results:
    print(f"  {r['name']:<24} ppl={r['val_ppl']:.1f}  tokens/seq={r['real_tokens_per_seq']:.0f}")
print("\n=== MODULO 2 COMPLETO ===")
print("Notebook 05: Tokenizacion comparada (k-mer / BPE)")
print("Notebook 06: H-Net + impacto downstream (este notebook)")
print("\nProximo: Modulo 3 - Pre-entrenamiento escalado y arquitecturas (SSM/Mamba)")